# Arc en Ciel Link 2.4.0 — notebook setup

Use this after installing a supported host and its dependencies. Your host must be a valid Git checkout with its launcher. Choose a runtime suitable for your host; consult [Colab runtime availability](https://research.google.com/colaboratory/faq.html). This notebook installs only Link, not the host or models.

Create a dedicated Link Key on arcenciel.io with jobs/inventory scopes. Add `ARCENCIEL_LINK_KEY` to Colab Secrets and enable notebook access. Never paste the key into a cell. Keep each concurrently running worker on its own key.


In [ ]:
import importlib.util
from pathlib import Path
from urllib.request import urlopen

# @title 1. Select an installed host and load the versioned setup helper
host_kind = "webui"  # @param ["webui", "comfyui", "swarmui"]
host_root = "/content/Forge-Neo"  # @param {type:"string"}

ARC_LINK_READY = False
helper_path = Path("/content/arcenciel_link_runtime_v2_4_0.py")
with urlopen(
    "https://raw.githubusercontent.com/FallenIncursio/arcenciel-link-webui/v2.4.0/notebooks/link_runtime.py", timeout=30
) as response:
    helper_path.write_bytes(response.read())
spec = importlib.util.spec_from_file_location("arcenciel_notebook_link", helper_path)
link = importlib.util.module_from_spec(spec)
spec.loader.exec_module(link)
link.validate_host(host_kind, host_root)
print("Host checkout and launcher verified.")

In [ ]:
# @title 2. Install Link and configure the secret before starting the host
link_key = link.read_colab_key()
link.install_extension(host_kind, host_root)
link.configure_runtime(host_kind, host_root, link_key)
del link_key
ARC_LINK_READY = True

### 3. Start your host

Run your host notebook's normal launch cell **after** the Link setup cell. It must inherit the environment of this notebook. If the host is already running, restart it. Keep the Link bridge on loopback; no public tunnel is needed for Link. If your host occupies a blocking launch cell, put the verification below in a background thread before that launch, or check the website's Link panel.

On arcenciel.io, save the same key, choose **Remote / Colab**, and press **Connect**. Pause/resume affects the current process; restarting reapplies the configured enabled state.


In [ ]:
# @title 4. Verify the authenticated worker (after the host has started)
if not globals().get("ARC_LINK_READY", False):
    raise RuntimeError("Run and complete the Link setup cells first")
link.wait_for_worker()

### Verify a download

Queue one small model using this key. Confirm DONE in Link history, the correct host folder, and the file's SHA-256. An online status alone does not verify the transfer. Stop the host and disconnect/delete the runtime when finished. Keys remain in runtime memory; sharing this notebook does not share those values.
